# NRI TBML Detection (Notebook 1 of 2)
- This notebook scores Canadian Non-Resident Importer (NRI) import declarations for **trade-based money-laundering (TBML) risk**.
- It builds shared reference and "foundation" tables, applies eight TBML indicators drawn from FATF typologies, and produces a per-importer **risk-profile matrix** that ranks importers by how many indicators they trip.

**What it produces**
- A per-transaction value table (CAD and declared), plus a per-importer feature table - the shared spine that Notebook 2 also uses.
- Eight TBML indicator flags per importer, combined into one risk score.

**Key conventions**
- An importer is identified by its 9-digit business number (`BN_9`) and a transaction by `CLNT_SPLY_REQ_ID`; names are unreliable and are not used for identity.
- Most monetary analytics use **CAD-normalized** values (`TXN_VALUE_CAD`, entity totals, I8 dormancy dates paired with CAD first-txn for I10) so figures are comparable across currencies.
- **Exceptions (not CAD-normalized):** I9 Rounded uses line-level declared `PRICE_VAL_TOTAL`; I4/I5 Benford (official) use transaction-level declared `TXN_VALUE_DECL` in native currency; I3 numbered-name and I7 jurisdiction checks are not value-based.
- Heavy computation runs inside the database; only small per-importer summaries come back into Python.

**Operating notes**
- `REBUILD` (see Configuration) switches the expensive table-building steps on or off, so the notebook can be re-run for analysis without rebuilding everything.
- The source is ~43 million rows.
- `ROW_FILTER` is a manual knob: at its default `1=1` (always true) it processes every row, so this run covers the full dataset.
- To test first, set it to a predicate (e.g. a one-month date range), run, then put it back to `1=1` for the full build.

## Setup
- Utility functions, the database connection, and configuration - plumbing, not analysis.
- A reader focused on results can skip to "Reference data".

### Helper functions
- Display and number-formatting helpers, plus a safe "drop table if it exists" used when rebuilding.

In [39]:
import pandas as pd
import numpy as np
import re
import os
import csv

def display_full(df):
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None, "display.width", 1000):
        display(df)

def barrier():
    print("\n<<", "=" * 50, ">>\n")

def drop_table_if_exists(table):
    try:
        idadb.ida_query(f"DROP TABLE {table};")
    except Exception:
        pass

In [40]:
def try_parse_number(value):
    if isinstance(value, (int, float)):
        return value
    if isinstance(value, str):
        if value.replace(",", "").replace(".", "", 1).isdigit():
            try:
                return float(value.replace(",", ""))
            except ValueError:
                return value
    return value

def format_df_numbers(df):
    df_copy = df.copy()
    def apply_formatting(value):
        if pd.isna(value):
            return value
        num = try_parse_number(value)
        if isinstance(num, (int, float)):
            if float(num).is_integer():
                return f"{int(num):,}"
            return f"{num:,.2f}"
        return value
    return df_copy.map(apply_formatting)

def format_specific_columns(df, columns_to_format):
    df_copy = df.copy()
    def apply_formatting(value):
        if pd.isna(value):
            return value
        num = try_parse_number(value)
        if isinstance(num, (int, float)):
            if float(num).is_integer():
                return f"{int(num):,}"
            return f"{num:,.2f}"
        return value
    for col in columns_to_format:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].apply(apply_formatting)
    return df_copy

### Database connection
- Opens the connection to the Netezza database that holds the import data.
- Credentials are read from a local `password` file.

In [41]:
from password import password
from nzpyida import IdaDataBase, IdaDataFrame

nzpy_dsn = {
    "database": "OADM_DEV",
    "port": 5480,
    "host": "EC63AP5692",
    "securityLevel": 1,
    "logLevel": 0
}

idadb = IdaDataBase(nzpy_dsn, uid="GMF939", pwd=password)

### Configuration
- Table names, file locations, and tuning knobs in one place.
- **Table names:** every output is written to the `OADM` schema with a `TBML_` prefix.
- **`REBUILD`:** `True` rebuilds all database tables from scratch (slow, full run); `False` reuses existing tables and just re-runs the analysis on top (fast).
- **`ROW_FILTER`:** restricts the source scan to a subset (e.g. one month) for quick tests; `"1=1"` means no filter - use everything.

In [42]:
REBUILD = True

SOURCE_TABLE = "OADM.TBML_T_NRI_IMPORT_M"
FX_TABLE = "OADM.TBML_T_CURRENCY_CONV_TO_CAD_2023_TO_2026_M"
FX_LONG_TABLE = "OADM.TBML_T_NRI_FX_LONG_M"
SECTION_MAP_TABLE = "OADM.TBML_T_NRI_HS_SECTION_MAP_M"
BASEL_TABLE = "OADM.TBML_T_NRI_BASEL_INDEX_M"
COUNTRY_TABLE = "OADM.TBML_T_NRI_COUNTRY_CODES_M"
TXN_TABLE = "OADM.TBML_T_NRI_TXN_VALUE_M"
CONSOLIDATOR_TABLE = "OADM.TBML_T_NRI_CONSOLIDATOR_M"
FEATURES_TABLE = "OADM.TBML_T_NRI_ENTITY_FEATURES_M"
IND_SRC_TABLE = "OADM.TBML_T_NRI_IND_SRC_M"
IND_TXN_TABLE = "OADM.TBML_T_NRI_IND_TXN_M"
RISK_MATRIX_TABLE = "OADM.TBML_T_NRI_RISK_MATRIX_M"

REF_DIR = "data_raw"
TEMP_DIR = "data_temp"
SECTION_FILE = REF_DIR + "/section to chapter.txt"
BASEL_FILE = REF_DIR + "/basel index.xlsx"
COUNTRY_FILE = REF_DIR + "/country codes.xlsx"

BASEL_HIGH_RISK_MIN = 5.5
CONSOLIDATOR_MIN_NAMES = 100
ROW_FILTER = "1=1"

### Bulk-load helper
- Plumbing for writing a Python table back into the database quickly and without data-type errors - not part of the analysis, safe to skip.
- It stages the data as a CSV, creates the database table with explicit column types, then bulk-loads it; a slower fallback method is kept below in case the fast path is unavailable.

In [43]:
def infer_nz_types(df):
    types = {}
    for c in df.columns:
        dt = df[c].dtype
        if pd.api.types.is_bool_dtype(dt):
            types[c] = "BOOLEAN"
        elif pd.api.types.is_integer_dtype(dt):
            types[c] = "BIGINT"
        elif pd.api.types.is_float_dtype(dt):
            types[c] = "DOUBLE"
        else:
            max_len = int(df[c].astype(str).str.len().max() or 0)
            width = max(8, min(4000, max_len * 2 + 8))
            types[c] = f"VARCHAR({width})"
    return types

def df_to_netezza(idadb, df, table_name, col_types=None, temp_dir=TEMP_DIR, delimiter=",", remotesource="PYTHON"):
    os.makedirs(temp_dir, exist_ok=True)
    if col_types is None:
        col_types = infer_nz_types(df)
    cols = list(col_types.keys())
    out = df[cols].copy()
    file_name = table_name.split(".")[-1] + ".csv"
    csv_path = os.path.abspath(os.path.join(temp_dir, file_name)).replace("\\", "/")
    log_dir = os.path.abspath(temp_dir).replace("\\", "/")
    out.to_csv(csv_path, index=False, sep=delimiter, na_rep="", quoting=csv.QUOTE_MINIMAL, encoding="utf-8")
    drop_table_if_exists(table_name)
    cols_ddl = ", ".join(f'"{c}" {t}' for c, t in col_types.items())
    idadb.ida_query(f"CREATE TABLE {table_name} ({cols_ddl}) DISTRIBUTE ON RANDOM;")
    load = f"""
    INSERT INTO {table_name}
    SELECT * FROM EXTERNAL '{csv_path}'
    USING (
        DELIMITER '{delimiter}'
        SKIPROWS 1
        QUOTEDVALUE DOUBLE
        NULLVALUE ''
        REMOTESOURCE '{remotesource}'
        LOGDIR '{log_dir}'
    );
    """
    idadb.ida_query(load)
    cnt = idadb.ida_query(f"SELECT COUNT(*) AS N FROM {table_name};")
    row_count = int(np.ravel(np.asarray(cnt))[0])
    print("Bulk-loaded", table_name, "->", row_count, "rows from", csv_path)
    return col_types

In [44]:
# Fallback only: chunked append via as_idadataframe. Use if df_to_netezza's external-table
# path is unavailable. chunk_size is auto-capped so rows * columns stays under Netezza's
# UNION-ALL cell budget (~10,000), which is what triggers the "Maximum number of union
# threshold" errors on wide tables.
from tqdm import tqdm

def prepare_df_for_netezza(df):
    df = df.copy()
    for col in df.columns:
        if "date" in col.lower():
            try:
                df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d").astype(object)
            except Exception:
                pass
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype.name == "str":
            df[col] = df[col].astype(object)
    return df

def upload_in_chunks(idadb, df, table_name, chunk_size=3000, clear_existing=True, union_cell_budget=9000):
    df = prepare_df_for_netezza(df)
    n_cols = max(1, df.shape[1])
    effective = max(1, min(chunk_size, union_cell_budget // n_cols))
    sample_df = df.iloc[[0]].copy().fillna("")
    idadf = idadb.as_idadataframe(sample_df, table_name, clear_existing=clear_existing)
    n = len(df)
    for start in tqdm(range(0, n, effective), desc=f"Uploading to {table_name}"):
        chunk = df.iloc[start:start + effective]
        if chunk.empty:
            continue
        idadb.append(idadf, chunk)
    return idadf

## Reference data
- External lookups the indicators rely on: currency exchange rates, the Harmonized System (HS) commodity hierarchy, the Basel anti-money-laundering risk index, and country codes.

### Currency coverage check
- Confirms the exchange-rate table's date range before we use it, so we know the rates actually cover the import dates we need to convert.

In [45]:
q = f"""
SELECT COUNT(*) AS n_rows, MIN(DATE) AS min_date, MAX(DATE) AS max_date
FROM {FX_TABLE};
"""
display_full(format_df_numbers(idadb.ida_query(q)))

,N_ROWS,MIN_DATE,MAX_DATE
0,827,2023-01-03,2026-04-27


In [46]:
q = f"""
SELECT *
FROM {FX_TABLE}
LIMIT 5;
"""
display_full(format_df_numbers(idadb.ida_query(q)))

,DATE,FXAUDCAD,FXBRLCAD,FXCNYCAD,FXEURCAD,FXHKDCAD,FXINRCAD,FXIDRCAD,FXJPYCAD,FXMYRCAD,FXMXNCAD,FXNZDCAD,FXNOKCAD,FXPENCAD,FXRUBCAD,FXSARCAD,FXSGDCAD,FXZARCAD,FXKRWCAD,FXSEKCAD,FXCHFCAD,FXTWDCAD,FXTHBCAD,FXTRYCAD,FXGBPCAD,FXUSDCAD
0,2024-02-08,0.87,0.27,0.19,1.45,0.17,0.02,8.6e-05,0.01,,0.08,0.82,0.13,0.35,0.01,0.36,1.00,0.07,0.00,0.13,1.54,0.04,,0.04,1.70,1.35
1,2024-07-08,0.92,0.25,0.19,1.48,0.17,0.02,8.4e-05,0.01,,0.08,0.84,0.13,0.36,0.02,0.36,1.01,0.08,0.00,0.13,1.52,0.04,,0.04,1.75,1.36
2,2024-05-09,0.90,0.27,0.19,1.47,0.18,0.02,8.5e-05,0.01,,0.08,0.82,0.13,0.37,0.01,0.37,1.01,0.07,0.00,0.13,1.51,0.04,,0.04,1.71,1.37
3,2023-12-05,0.89,0.27,0.19,1.47,0.17,0.02,8.8e-05,0.01,,0.08,0.83,0.12,0.36,0.01,0.36,1.01,0.07,0.00,0.13,1.55,0.04,,0.05,1.71,1.36
4,2025-06-17,0.88,0.25,0.19,1.57,0.17,0.02,8.4e-05,0.01,,0.07,0.82,0.14,0.38,0.02,0.36,1.06,0.08,0.00,0.14,1.67,0.05,,0.03,1.84,1.36


### Currency - exchange rates for date-matching
- Reshapes the exchange-rate table to one row per (date, currency, rate-to-CAD) so each transaction can be matched to the rate in effect on its arrival date.
- `CAD` is added at a rate of 1.0 (no conversion).
- A few currencies have no published rate; transactions in those are left unconverted and counted separately rather than silently dropped.

In [47]:
fx_wide = idadb.ida_query(f"SELECT * FROM {FX_TABLE};")
fx_wide.columns = [c.upper() for c in fx_wide.columns]
fx_wide["FX_DATE"] = pd.to_datetime(fx_wide["DATE"]).dt.strftime("%Y-%m-%d")
fx_cols = [c for c in fx_wide.columns if c.startswith("FX") and c.endswith("CAD")]
fx_long = fx_wide.melt(id_vars="FX_DATE", value_vars=fx_cols, var_name="FX_COL", value_name="RATE_TO_CAD")
fx_long["CCY"] = fx_long["FX_COL"].str[2:5]
fx_long = fx_long[fx_long["RATE_TO_CAD"].notna() & (fx_long["RATE_TO_CAD"].astype(str).str.strip() != "")]
fx_long = fx_long[["FX_DATE", "CCY", "RATE_TO_CAD"]]
cad = pd.DataFrame({"FX_DATE": sorted(fx_wide["FX_DATE"].unique())})
cad["CCY"] = "CAD"
cad["RATE_TO_CAD"] = 1.0
fx_long = pd.concat([fx_long, cad], ignore_index=True)
fx_long["RATE_TO_CAD"] = fx_long["RATE_TO_CAD"].astype(float)
print("Fx_long rows:", len(fx_long), "| currencies:", sorted(fx_long["CCY"].unique()))

Fx_long rows: 19848 | currencies: ['AUD', 'BRL', 'CAD', 'CHF', 'CNY', 'EUR', 'GBP', 'HKD', 'IDR', 'INR', 'JPY', 'KRW', 'MXN', 'NOK', 'NZD', 'PEN', 'RUB', 'SAR', 'SEK', 'SGD', 'TRY', 'TWD', 'USD', 'ZAR']


In [48]:
if REBUILD:
    df_to_netezza(idadb, fx_long, FX_LONG_TABLE, col_types={
        "FX_DATE": "VARCHAR(10)",
        "CCY": "VARCHAR(8)",
        "RATE_TO_CAD": "DOUBLE",
    })
display_full(idadb.ida_query(f"SELECT * FROM {FX_LONG_TABLE} ORDER BY CCY, FX_DATE LIMIT 8;"))

Bulk-loaded OADM.TBML_T_NRI_FX_LONG_M -> 19848 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_FX_LONG_M.csv


,FX_DATE,CCY,RATE_TO_CAD
0,2023-01-03,AUD,0.9196
1,2023-01-04,AUD,0.9256
2,2023-01-05,AUD,0.9171
3,2023-01-06,AUD,0.9220
4,2023-01-09,AUD,0.9274
5,2023-01-10,AUD,0.9241
6,2023-01-11,AUD,0.9265
7,2023-01-12,AUD,0.9303


### HS commodity sections (from file)
- Maps each 2-digit HS chapter to its broader HS "section".
- This lets us measure how many distinct *categories of goods* an importer deals in - the basis for the "inconsistent lines of trade" indicator.

In [49]:
def parse_section_map(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = re.match(r"Section\s+([IVXL]+):\s+Chapters?\s+(\d+)(?:\s*-\s*(\d+))?", line)
            if not m:
                continue
            section = m.group(1)
            start = int(m.group(2))
            end = int(m.group(3)) if m.group(3) else start
            for chap in range(start, end + 1):
                rows.append({"HS_CHAP_NBR": f"{chap:02d}", "HS_SECTION": section})
    return pd.DataFrame(rows)

section_map = parse_section_map(SECTION_FILE)
print("section_map rows:", len(section_map))
display_full(section_map.head(10))
if REBUILD:
    df_to_netezza(idadb, section_map, SECTION_MAP_TABLE, col_types={
        "HS_CHAP_NBR": "VARCHAR(2)",
        "HS_SECTION": "VARCHAR(8)",
    })

section_map rows: 97


,HS_CHAP_NBR,HS_SECTION
0,01,I
1,02,I
2,03,I
3,04,I
4,05,I
5,06,II
6,07,II
7,08,II
8,09,II
9,10,II


Bulk-loaded OADM.TBML_T_NRI_HS_SECTION_MAP_M -> 97 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_HS_SECTION_MAP_M.csv


### Basel AML index (from file)
- Loads the Basel Anti-Money-Laundering Index, a country-level risk score.
- Countries at or above the configured threshold are marked high-risk, which feeds the high-risk-jurisdiction indicator.

In [50]:
basel = pd.read_excel(BASEL_FILE, sheet_name="Basel Index")
basel = basel[["ISO Code", "Country", "Basel_ML_TF_RISK"]].copy()
basel = basel.rename(columns={"ISO Code": "ISO_CODE", "Country": "COUNTRY", "Basel_ML_TF_RISK": "BASEL_ML_TF_RISK"})
basel["ISO_CODE"] = basel["ISO_CODE"].astype(str).str.strip().str.upper()
basel["BASEL_ML_TF_RISK"] = basel["BASEL_ML_TF_RISK"].astype(float)
basel["HIGH_RISK_IND"] = (basel["BASEL_ML_TF_RISK"] >= BASEL_HIGH_RISK_MIN).astype(int)
print("basel rows:", len(basel), "| high-risk countries:", int(basel["HIGH_RISK_IND"].sum()))
display_full(basel.head(8))
if REBUILD:
    df_to_netezza(idadb, basel, BASEL_TABLE, col_types={
        "ISO_CODE": "VARCHAR(3)",
        "COUNTRY": "VARCHAR(128)",
        "BASEL_ML_TF_RISK": "DOUBLE",
        "HIGH_RISK_IND": "INTEGER",
    })

basel rows: 254 | high-risk countries: 97


,ISO_CODE,COUNTRY,BASEL_ML_TF_RISK,HIGH_RISK_IND
0,AF,Afghanistan,8.39,1
1,AL,Albania,5.23,0
2,DZ,Algeria,7.05,1
3,AD,Andorra,4.10,0
4,AO,Angola,6.97,1
5,AI,Anguilla,5.31,0
6,AG,Antigua and Barbuda,5.78,1
7,AR,Argentina,5.13,0


Bulk-loaded OADM.TBML_T_NRI_BASEL_INDEX_M -> 254 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_BASEL_INDEX_M.csv


### Country codes (from file)
- Country names and world-region labels, loaded mainly for readable reporting.
- **Caveat:** this file's country `Code` uses a CBSA-style scheme (it even splits the US into states) that does **not** match the ISO-2 vendor country codes in the import data.
- Use it for naming/region rollups only - the high-risk check uses the Basel ISO codes directly, not this file.

In [51]:
country = pd.read_excel(COUNTRY_FILE, sheet_name="Country Codes")
country = country.rename(columns={"CountryOfOriginName": "COUNTRY_NAME", "Code": "COUNTRY_CODE", "RegionWorld": "REGION_WORLD", "SubRegion": "SUB_REGION"})
country = country[["COUNTRY_NAME", "COUNTRY_CODE", "REGION_WORLD", "SUB_REGION"]].copy()
country["COUNTRY_CODE"] = country["COUNTRY_CODE"].astype(str).str.strip().str.upper()
print("country rows:", len(country))
display_full(country.head(8))
if REBUILD:
    df_to_netezza(idadb, country, COUNTRY_TABLE, col_types={
        "COUNTRY_NAME": "VARCHAR(128)",
        "COUNTRY_CODE": "VARCHAR(8)",
        "REGION_WORLD": "VARCHAR(64)",
        "SUB_REGION": "VARCHAR(64)",
    })

country rows: 300


,COUNTRY_NAME,COUNTRY_CODE,REGION_WORLD,SUB_REGION
0,Alabama,UAL,Americas,Northern America
1,Alaska,UAK,Americas,Northern America
2,Arizona,UAZ,Americas,Northern America
3,Arkansas,UAR,Americas,Northern America
4,California,UCA,Americas,Northern America
5,Colorado,UCO,Americas,Northern America
6,Connecticut,UCT,Americas,Northern America
7,Delaware,UDE,Americas,Northern America


Bulk-loaded OADM.TBML_T_NRI_COUNTRY_CODES_M -> 300 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_COUNTRY_CODES_M.csv


In [52]:
q = f"""
SELECT
COUNT(DISTINCT s.VENDOR_CNTRY_CDE) AS distinct_vendor_cntry,
COUNT(DISTINCT b.ISO_CODE) AS matched_basel_codes
FROM {SOURCE_TABLE} s
LEFT JOIN {BASEL_TABLE} b ON b.ISO_CODE = s.VENDOR_CNTRY_CDE
WHERE s.VENDOR_CNTRY_CDE IS NOT NULL AND s.VENDOR_CNTRY_CDE <> 'NULL' AND s.VENDOR_CNTRY_CDE <> '';
"""
display_full(format_df_numbers(idadb.ida_query(q)))

,DISTINCT_VENDOR_CNTRY,MATCHED_BASEL_CODES
0,134,128


**Interpretation - coverage check**
- 128 of the 134 distinct vendor country codes match a Basel country, so the high-risk-jurisdiction indicator (I7) has near-complete coverage.
- The 6 unmatched codes default to low-risk via `COALESCE` on the left join.
- To list them and their volume, run a `LEFT JOIN` from the source vendor country codes to the Basel table and filter where Basel is null (see `FOLLOWUPS.md`).

## Foundation tables
- Three tables built once and reused by both notebooks: transaction value (CAD + declared), a consolidator flag, and a per-importer feature summary.

### Transaction value (CAD + declared)
- Each import line carries a price in its own currency.
- Match every line to the most recent exchange rate on or before its arrival date, convert to CAD, and sum to `TXN_VALUE_CAD` per transaction (`CLNT_SPLY_REQ_ID`).
- Also store `TXN_VALUE_DECL`: the declared-currency transaction total, summed only when **all lines share one currency** (`DECL_CCY`).
- Mixed-currency transactions leave `TXN_VALUE_DECL` null and set `IS_MIXED_CCY_TXN = 1` (excluded from official Benford).
- Keep `LINE_CNT` (lines per transaction) and `LINES_FX_MISSING` (lines we could not convert to CAD).
- **NOTE:** amended/re-transmitted declarations (`REQ_VERS_NBR`) are not yet de-duplicated, so a transaction filed more than once can still be over-counted - a known refinement for later.

In [53]:
if REBUILD:
    drop_table_if_exists(TXN_TABLE)
    q = f"""
    CREATE TABLE {TXN_TABLE} AS
    WITH base AS (
        SELECT
            s.CLNT_SPLY_REQ_ID,
            s.BN_9,
            SUBSTR(CAST(s.ACTUAL_ARRIVAL_DTTM AS VARCHAR(30)), 1, 10) AS ARRIVAL_DATE,
            s.PRICE_VAL_TOTAL,
            s.PRICE_CRNCY_TOTAL,
            CAST(NULLIF(NULLIF(TRIM(s.PRICE_VAL_TOTAL), 'NULL'), '') AS DOUBLE) AS PRICE_VAL
        FROM {SOURCE_TABLE} s
        WHERE {ROW_FILTER}
    ),
    txn_ccy AS (
        SELECT
            CLNT_SPLY_REQ_ID,
            COUNT(DISTINCT PRICE_CRNCY_TOTAL) AS CCY_CNT,
            MAX(PRICE_CRNCY_TOTAL) AS DECL_CCY
        FROM base
        GROUP BY CLNT_SPLY_REQ_ID
    ),
    pairs AS (
        SELECT DISTINCT PRICE_CRNCY_TOTAL AS CCY, ARRIVAL_DATE FROM base
    ),
    asof AS (
        SELECT CCY, ARRIVAL_DATE, RATE_TO_CAD FROM (
            SELECT
                p.CCY,
                p.ARRIVAL_DATE,
                f.RATE_TO_CAD,
                ROW_NUMBER() OVER (PARTITION BY p.CCY, p.ARRIVAL_DATE ORDER BY f.FX_DATE DESC) AS rn
            FROM pairs p
            LEFT JOIN {FX_LONG_TABLE} f
                ON f.CCY = p.CCY AND f.FX_DATE <= p.ARRIVAL_DATE
        ) z
        WHERE rn = 1
    )
    SELECT
        b.CLNT_SPLY_REQ_ID,
        MAX(b.BN_9) AS BN_9,
        MAX(b.ARRIVAL_DATE) AS ARRIVAL_DATE,
        SUM(CASE
                WHEN a.RATE_TO_CAD IS NOT NULL
                THEN b.PRICE_VAL * a.RATE_TO_CAD
            END) AS TXN_VALUE_CAD,
        SUM(CASE
                WHEN tc.CCY_CNT = 1 AND b.PRICE_VAL IS NOT NULL
                THEN b.PRICE_VAL
            END) AS TXN_VALUE_DECL,
        MAX(CASE WHEN tc.CCY_CNT = 1 THEN tc.DECL_CCY END) AS DECL_CCY,
        MAX(CASE WHEN tc.CCY_CNT > 1 THEN 1 ELSE 0 END) AS IS_MIXED_CCY_TXN,
        SUM(CASE WHEN a.RATE_TO_CAD IS NULL THEN 1 ELSE 0 END) AS LINES_FX_MISSING,
        COUNT(*) AS LINE_CNT
    FROM base b
    JOIN txn_ccy tc ON tc.CLNT_SPLY_REQ_ID = b.CLNT_SPLY_REQ_ID
    LEFT JOIN asof a
        ON a.CCY = b.PRICE_CRNCY_TOTAL AND a.ARRIVAL_DATE = b.ARRIVAL_DATE
    GROUP BY b.CLNT_SPLY_REQ_ID
    DISTRIBUTE ON (CLNT_SPLY_REQ_ID);
    """
    idadb.ida_query(q)
    print("Created", TXN_TABLE)

Created OADM.TBML_T_NRI_TXN_VALUE_M


In [54]:
q = f"""
SELECT
COUNT(*) AS n_transactions,
COUNT(DISTINCT BN_9) AS n_entities,
SUM(CASE WHEN LINES_FX_MISSING > 0 THEN 1 ELSE 0 END) AS txns_with_fx_gaps,
SUM(CASE WHEN IS_MIXED_CCY_TXN = 1 THEN 1 ELSE 0 END) AS txns_mixed_ccy,
SUM(CASE WHEN TXN_VALUE_DECL IS NOT NULL THEN 1 ELSE 0 END) AS txns_with_decl_value,
MIN(LINE_CNT) AS min_lines,
AVG(LINE_CNT) AS avg_lines,
MAX(LINE_CNT) AS max_lines,
SUM(TXN_VALUE_CAD) AS total_value_cad
FROM {TXN_TABLE};
"""
display_full(format_df_numbers(idadb.ida_query(q)))

,N_TRANSACTIONS,N_ENTITIES,TXNS_WITH_FX_GAPS,TXNS_MIXED_CCY,TXNS_WITH_DECL_VALUE,MIN_LINES,AVG_LINES,MAX_LINES,TOTAL_VALUE_CAD
0,"3,786,520","19,366","2,782","25,220","3,761,294",1,11.48,"22,374","80,530,979,458.84"


**Interpretation - transaction value table**
- ~3.79M transactions across ~19,400 importers, totalling ~$80.5B CAD.
- Average ~11.5 lines per transaction (max ~22,000) confirms transactions are correctly assembled from multiple lines.
- ~2,800 transactions include at least one line in a currency we could not convert (`LINES_FX_MISSING`), so their CAD total may be partial or null.
- ~25,220 transactions are mixed-currency (`IS_MIXED_CCY_TXN = 1`); ~3.76M have a usable declared total (`TXN_VALUE_DECL`).
- Official Benford (I4/I5) uses only single-currency transactions with a populated `TXN_VALUE_DECL`.

In [55]:
q = f"""
SELECT *
FROM {TXN_TABLE}
LIMIT 5;
"""
display_full(format_specific_columns(idadb.ida_query(q), ["TXN_VALUE_CAD", "TXN_VALUE_DECL"]))

,CLNT_SPLY_REQ_ID,BN_9,ARRIVAL_DATE,TXN_VALUE_CAD,TXN_VALUE_DECL,DECL_CCY,IS_MIXED_CCY_TXN,LINES_FX_MISSING,LINE_CNT
0,10027000808678,809990690,2025-11-08,"20,668.98","455,565",TWD,0,0,1
1,62166885511162,814492823,2025-05-04,"10,312.94","54,335.82",CNY,0,0,4
2,13003232636281,869541672,2025-12-20,674.31,417.48,EUR,0,0,2
3,16865320920795,789846623,2025-01-22,911.92,"4,615",CNY,0,0,2
4,12021015461180,868166018,2025-01-16,"33,813.57","22,828.50",EUR,0,0,1


### Consolidators / couriers
- Some `BN_9`s are marketplaces or couriers (Amazon-type accounts) that file on behalf of thousands of unrelated importers, so they naturally look extremely "diverse" and would set off certain indicators for the wrong reason.
- Any importer whose count of distinct importer names exceeds a threshold is flagged as a consolidator.
- **I9 (Rounded)** and **I11 (Lines of Trade)** are suppressed for consolidators.

In [56]:
CONSOLIDATOR_MIN_NAMES

100

In [57]:
if REBUILD:
    drop_table_if_exists(CONSOLIDATOR_TABLE)
    q = f"""
    CREATE TABLE {CONSOLIDATOR_TABLE} AS
    SELECT
        BN_9,
        COUNT(DISTINCT IMPORTER_NAME) AS DISTINCT_IMPORTER_NAMES,
        COUNT(DISTINCT CONSIGNEE_NAME) AS DISTINCT_CONSIGNEES,
        COUNT(DISTINCT CLNT_SPLY_REQ_ID) AS TXN_CNT,
        CASE WHEN COUNT(DISTINCT IMPORTER_NAME) >= {CONSOLIDATOR_MIN_NAMES} THEN 1 ELSE 0 END AS IS_CONSOLIDATOR
    FROM {SOURCE_TABLE}
    WHERE {ROW_FILTER}
    GROUP BY BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("Created", CONSOLIDATOR_TABLE)

Created OADM.TBML_T_NRI_CONSOLIDATOR_M


In [58]:
q = f"""
SELECT
COUNT(*) AS n_entities,
SUM(IS_CONSOLIDATOR) AS n_consolidators
FROM {CONSOLIDATOR_TABLE};
"""
display_full(format_df_numbers(idadb.ida_query(q)))

,N_ENTITIES,N_CONSOLIDATORS
0,"19,367",5


**Interpretation - consolidators**
- Only 5 of ~19,400 importers cross the consolidator threshold (marketplace/courier accounts).
- I9 and I11 are suppressed for these so they do not dominate the rankings.

In [59]:
q = f"""
SELECT *
FROM {CONSOLIDATOR_TABLE}
WHERE IS_CONSOLIDATOR = 1;
"""
idadb.ida_query(q)

,BN_9,DISTINCT_IMPORTER_NAMES,DISTINCT_CONSIGNEES,TXN_CNT,IS_CONSOLIDATOR
0,725844757,108813,0,122232,1
1,723939757,45015,0,48325,1
2,726799752,117727,0,131240,1
3,725416754,70428,0,78112,1
4,763031622,41214,0,43930,1


### Per-importer feature table
- Collapses everything to one row per importer: transaction count and value, how many distinct commodities/countries/vendors it deals with, and the consolidator flag.
- This compact table is what the indicators and the risk matrix are built from.

In [60]:
if REBUILD:
    drop_table_if_exists(FEATURES_TABLE)
    q = f"""
    CREATE TABLE {FEATURES_TABLE} AS
    SELECT
        a.BN_9,
        a.TXN_CNT,
        a.TOTAL_VALUE_CAD,
        a.AVG_TXN_VALUE_CAD,
        a.MAX_TXN_VALUE_CAD,
        a.FIRST_ARRIVAL,
        a.LAST_ARRIVAL,
        b.LINE_CNT,
        b.DISTINCT_HS6,
        b.DISTINCT_CHAPTERS,
        b.DISTINCT_SECTIONS,
        b.DISTINCT_VENDORS,
        b.DISTINCT_VENDOR_COUNTRIES,
        COALESCE(c.IS_CONSOLIDATOR, 0) AS IS_CONSOLIDATOR
    FROM (
        SELECT
            BN_9,
            COUNT(*) AS TXN_CNT,
            SUM(TXN_VALUE_CAD) AS TOTAL_VALUE_CAD,
            AVG(TXN_VALUE_CAD) AS AVG_TXN_VALUE_CAD,
            MAX(TXN_VALUE_CAD) AS MAX_TXN_VALUE_CAD,
            MIN(ARRIVAL_DATE) AS FIRST_ARRIVAL,
            MAX(ARRIVAL_DATE) AS LAST_ARRIVAL
        FROM {TXN_TABLE}
        GROUP BY BN_9
    ) a
    LEFT JOIN (
        SELECT
            s.BN_9,
            COUNT(*) AS LINE_CNT,
            COUNT(DISTINCT s.HS_CHAP_NBR || s.HS_HD_SFX || s.HS_SBHD_SFX) AS DISTINCT_HS6,
            COUNT(DISTINCT s.HS_CHAP_NBR) AS DISTINCT_CHAPTERS,
            COUNT(DISTINCT m.HS_SECTION) AS DISTINCT_SECTIONS,
            COUNT(DISTINCT s.VENDOR_NAME) AS DISTINCT_VENDORS,
            COUNT(DISTINCT s.VENDOR_CNTRY_CDE) AS DISTINCT_VENDOR_COUNTRIES
        FROM {SOURCE_TABLE} s
        LEFT JOIN {SECTION_MAP_TABLE} m ON m.HS_CHAP_NBR = s.HS_CHAP_NBR
        WHERE {ROW_FILTER}
        GROUP BY s.BN_9
    ) b ON b.BN_9 = a.BN_9
    LEFT JOIN {CONSOLIDATOR_TABLE} c ON c.BN_9 = a.BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("Created", FEATURES_TABLE)

Created OADM.TBML_T_NRI_ENTITY_FEATURES_M


In [61]:
q = f"""
SELECT *
FROM {FEATURES_TABLE}
ORDER BY TOTAL_VALUE_CAD DESC
LIMIT 15;
"""
display_full(format_specific_columns(idadb.ida_query(q), ["TXN_CNT", "TOTAL_VALUE_CAD", "AVG_TXN_VALUE_CAD", "MAX_TXN_VALUE_CAD", "LINE_CNT"]))

,BN_9,TXN_CNT,TOTAL_VALUE_CAD,AVG_TXN_VALUE_CAD,MAX_TXN_VALUE_CAD,FIRST_ARRIVAL,LAST_ARRIVAL,LINE_CNT,DISTINCT_HS6,DISTINCT_CHAPTERS,DISTINCT_SECTIONS,DISTINCT_VENDORS,DISTINCT_VENDOR_COUNTRIES,IS_CONSOLIDATOR
0,859073660,95,"9,122,015,276.17","96,021,213.43","1,134,447,019.20",2025-01-02,2025-12-26,"146,884",20,2,1,9,4,0
1,866118771,264,"3,922,979,739.72","14,859,771.74","632,417,048.14",2025-01-16,2025-12-29,"14,530",7,1,1,1,1,0
2,816276174,905,"1,507,232,572.72","1,665,450.36","41,477,389.75",2025-01-01,2025-12-31,"2,397",37,9,6,79,14,0
3,748335627,"1,151","1,264,253,286.38","1,098,395.56","32,691,067.15",2025-01-13,2025-12-29,"7,983",43,18,11,237,1,0
4,814492823,"23,970","1,263,545,835.25","58,281.63","762,163.51",2025-01-01,2025-12-31,"1,355,380",334,43,14,503,37,0
5,805593811,"8,646","1,220,446,979.65","141,157.41","644,322.73",2025-01-01,2025-12-31,"3,468,590",1153,71,19,13,2,0
6,890119852,"4,625","925,408,689.63","200,131.64","1,650,627.03",2025-01-02,2025-12-31,"1,851,734",306,29,14,30,1,0
7,138652060,30,"918,436,480.76","30,614,549.36","917,207,618.69",2025-01-02,2025-03-07,30,2,1,1,2,1,0
8,898415716,"16,010","894,850,119.44","55,893.20","552,303.26",2025-01-01,2025-12-31,"1,620,283",736,59,18,4,1,0
9,887302024,104,"848,833,632.58","8,161,861.85","100,859,734.86",2025-01-08,2025-12-30,"10,938",12,2,1,16,6,0


In [62]:
q = f"""
SELECT COUNT(*)
FROM {FEATURES_TABLE};
"""
idadb.ida_query(q)

0    19366
Name: COUNT, dtype: int64

## TBML indicators (first pass)
- Eight indicators, each producing a yes/no flag per importer, taken from the FATF/Egmont TBML indicator methodology.
- The IDs below are the document's own indicator numbers (scalable list 1-12, non-scalable 13-15); we implement only the subset computable from customs data, so numbering is intentionally non-contiguous (omitted indicators explained at the end).
- **I9 (Rounded)** and **I11 (Lines of Trade)** are suppressed for consolidators.

**Indicators**
- **I3 - Numbered Company:** importer/vendor names that are mostly leading digits (a shell-company signature).
- **I4 - Benford's Law (1st Digit):** first digits of declared transaction value deviate from the natural Benford distribution (transaction-level, single-currency only).
- **I5 - Benford's Law (2nd Digit):** the same test applied to the second digit.
- **I7 - Trades with High Risk Jurisdictions:** vendors in countries the Basel index rates high-risk (score 5.5 or above).
- **I8 - High Dormancy:** unusually long gaps between shipments.
- **I9 - Rounded Transactions:** an unusually high share of suspiciously round declared line values (exact multiples of 1,000 / 100).
- **I10 - High First Transaction:** an unusually large very first transaction (CAD percentile).
- **I11 - Inconsistent Lines of Trade:** an importer trading across more than one HS section (unrelated commodity categories).

**Omitted indicators (and why)**
- **I1 Miscalculations** and **I2 Inconsistent Prices** need reliable price-per-unit (invoice total / declared units); this data has no dependable quantity / UOM.
- **I6 One-to-One Relationship** is computable from this data; deferred for scope - a near-term candidate to add.
- **I12 Vague Descriptions** needs goods-description text analysis; the description field is free-text and noisy.
- **I13-I15 (non-scalable)** need company-registry data not present in this dataset.

### Indicator settings & Benford helpers
- Thresholds for each indicator, plus the expected Benford digit distributions used to score I4/I5.

In [63]:
NUMBERED_SHARE_MIN = 0.5
SECTIONS_FLAG_MIN = 3
BENFORD_MIN_TXNS = 50
MAD1_THRESHOLD = 0.015
MAD2_THRESHOLD = 0.012
FLAG_PCTL = 0.75
ROUND_SHARE_FLOOR = 0.10

INDICATOR_WEIGHTS = {
    "I3_NUMBERED": 1.0,
    "I4_BENFORD1": 1.0,
    "I5_BENFORD2": 1.0,
    "I7_HIGH_RISK_JURIS": 1.0,
    "I8_DORMANCY": 1.0,
    "I9_ROUNDED": 1.0,
    "I10_HIGH_FIRST_TXN": 1.0,
    "I11_LINES_OF_TRADE": 1.0,
}

def benford_first_digit_expected():
    d = np.arange(1, 10)
    return np.log10(1 + 1 / d)

def benford_second_digit_expected():
    out = []
    for d in range(0, 10):
        p = sum(np.log10(1 + 1 / (10 * k + d)) for k in range(1, 10))
        out.append(p)
    return np.array(out)

def mad_from_counts(counts, expected):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return np.nan
    observed = counts / total
    return float(np.mean(np.abs(observed - expected)))

### Indicators from the import lines (I3, I7, I9)
- Computed directly from the raw import lines, per importer (`BN_9`).
- **I3 (Numbered Company):** share of lines whose importer/vendor name starts with 4+ digits (avoids false hits like "3M").
- **I7 (Trades with High Risk Jurisdictions):** whether any vendor sits in a Basel high-risk country, plus the share of such lines.
- **I9 (Rounded Transactions):** share of lines whose original-currency `PRICE_VAL_TOTAL` is an exact multiple of 1,000 / 100 (tested before FX conversion).

In [64]:
if REBUILD:
    drop_table_if_exists(IND_SRC_TABLE)
    q = f"""
    CREATE TABLE {IND_SRC_TABLE} AS
    SELECT
        s.BN_9,
        AVG(
            CASE
                WHEN SUBSTR(LTRIM(s.IMPORTER_NAME), 1, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.IMPORTER_NAME), 2, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.IMPORTER_NAME), 3, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.IMPORTER_NAME), 4, 1) BETWEEN '0' AND '9'
                THEN 1.0 ELSE 0 END
        ) AS IMPORTER_NUMBERED_SHARE,
        AVG(
            CASE
                WHEN SUBSTR(LTRIM(s.VENDOR_NAME), 1, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.VENDOR_NAME), 2, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.VENDOR_NAME), 3, 1) BETWEEN '0' AND '9'
                 AND SUBSTR(LTRIM(s.VENDOR_NAME), 4, 1) BETWEEN '0' AND '9'
                THEN 1.0 ELSE 0 END
        ) AS VENDOR_NUMBERED_SHARE,
        AVG(
            CASE
                WHEN CAST(NULLIF(NULLIF(TRIM(s.PRICE_VAL_TOTAL), 'NULL'), '') AS DOUBLE) >= 1
                 AND MOD(CAST(ROUND(CAST(NULLIF(NULLIF(TRIM(s.PRICE_VAL_TOTAL), 'NULL'), '') AS DOUBLE)) AS BIGINT), 1000) = 0
                THEN 1.0 ELSE 0 END
        ) AS ROUND_1000_SHARE,
        AVG(
            CASE
                WHEN CAST(NULLIF(NULLIF(TRIM(s.PRICE_VAL_TOTAL), 'NULL'), '') AS DOUBLE) >= 1
                 AND MOD(CAST(ROUND(CAST(NULLIF(NULLIF(TRIM(s.PRICE_VAL_TOTAL), 'NULL'), '') AS DOUBLE)) AS BIGINT), 100) = 0
                THEN 1.0 ELSE 0 END
        ) AS ROUND_100_SHARE,
        MAX(COALESCE(b.HIGH_RISK_IND, 0)) AS ANY_HIGH_RISK,
        AVG(CASE WHEN COALESCE(b.HIGH_RISK_IND, 0) = 1 THEN 1.0 ELSE 0 END) AS HIGH_RISK_SHARE
    FROM {SOURCE_TABLE} s
    LEFT JOIN {BASEL_TABLE} b ON b.ISO_CODE = s.VENDOR_CNTRY_CDE
    WHERE {ROW_FILTER}
    GROUP BY s.BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("Created", IND_SRC_TABLE)

Created OADM.TBML_T_NRI_IND_SRC_M


### Indicators from the transactions (I8, I10)
- Per importer, from the transaction table.
- **I8 (High Dormancy):** the longest gap, in days, between consecutive shipments.
- **I10 (High First Transaction):** the CAD value of the importer's earliest transaction (population percentile).

In [65]:
q = f"""
SELECT *
FROM {TXN_TABLE}
LIMIT 5;
"""
idadb.ida_query(q)

,CLNT_SPLY_REQ_ID,BN_9,ARRIVAL_DATE,TXN_VALUE_CAD,TXN_VALUE_DECL,DECL_CCY,IS_MIXED_CCY_TXN,LINES_FX_MISSING,LINE_CNT
0,10093000585658,807599311,2025-07-14,1596.422394,866.82,GBP,0,0,4
1,62166885587061,814492823,2025-08-07,NaN,1339160.31,THB,0,5,5
2,62166885487754,814492823,2025-04-03,28687.045842,18435.22,EUR,0,0,5
3,62166885560071,814492823,2025-07-03,51770.895726,32441.97,EUR,0,0,14
4,62166885590442,814492823,2025-08-14,29960.095120,NaN,NaN,1,10,13


In [66]:
if REBUILD:
    drop_table_if_exists(IND_TXN_TABLE)
    q = f"""
    CREATE TABLE {IND_TXN_TABLE} AS
    SELECT
        BN_9,
        MAX(GAP_DAYS) AS MAX_GAP_DAYS,
        MAX(CASE WHEN RN_ASC = 1 THEN TXN_VALUE_CAD END) AS FIRST_TXN_VALUE_CAD
    FROM (
        SELECT
            BN_9,
            TXN_VALUE_CAD,
            CAST(ARRIVAL_DATE AS DATE) - LAG(CAST(ARRIVAL_DATE AS DATE)) OVER (PARTITION BY BN_9 ORDER BY CAST(ARRIVAL_DATE AS DATE)) AS GAP_DAYS,
            ROW_NUMBER() OVER (PARTITION BY BN_9 ORDER BY CAST(ARRIVAL_DATE AS DATE) ASC, CLNT_SPLY_REQ_ID) AS RN_ASC
        FROM {TXN_TABLE}
        WHERE ARRIVAL_DATE IS NOT NULL AND ARRIVAL_DATE <> 'NULL' AND ARRIVAL_DATE <> ''
    ) t
    GROUP BY BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("Created", IND_TXN_TABLE)

Created OADM.TBML_T_NRI_IND_TXN_M


In [67]:
q = f"""
SELECT *
FROM {IND_TXN_TABLE}
LIMIT 5;
"""
idadb.ida_query(q)

,BN_9,MAX_GAP_DAYS,FIRST_TXN_VALUE_CAD
0,121468730,11,194.643
1,127232882,10,19379.280
2,130412067,16,10320.000
3,703523217,67,13608.990
4,706444098,41,21375.920


### Assemble one row per importer
- Joins the feature table to the indicator inputs and pulls the result into Python - one row per importer, small enough to finish scoring in memory.

In [68]:
q = f"""
SELECT
    f.BN_9, f.TXN_CNT, f.TOTAL_VALUE_CAD, f.AVG_TXN_VALUE_CAD, f.MAX_TXN_VALUE_CAD,
    f.FIRST_ARRIVAL, f.LAST_ARRIVAL, f.LINE_CNT,
    f.DISTINCT_HS6, f.DISTINCT_CHAPTERS, f.DISTINCT_SECTIONS,
    f.DISTINCT_VENDORS, f.DISTINCT_VENDOR_COUNTRIES, f.IS_CONSOLIDATOR,
    s.IMPORTER_NUMBERED_SHARE, s.VENDOR_NUMBERED_SHARE,
    s.ROUND_1000_SHARE, s.ROUND_100_SHARE, s.ANY_HIGH_RISK, s.HIGH_RISK_SHARE,
    t.MAX_GAP_DAYS, t.FIRST_TXN_VALUE_CAD
FROM {FEATURES_TABLE} f
LEFT JOIN {IND_SRC_TABLE} s ON s.BN_9 = f.BN_9
LEFT JOIN {IND_TXN_TABLE} t ON t.BN_9 = f.BN_9;
"""
feat = idadb.ida_query(q)
feat.columns = [c.upper() for c in feat.columns]
print("Entities:", len(feat))
display_full(feat.head())

Entities: 19366


,BN_9,TXN_CNT,TOTAL_VALUE_CAD,AVG_TXN_VALUE_CAD,MAX_TXN_VALUE_CAD,FIRST_ARRIVAL,LAST_ARRIVAL,LINE_CNT,DISTINCT_HS6,DISTINCT_CHAPTERS,DISTINCT_SECTIONS,DISTINCT_VENDORS,DISTINCT_VENDOR_COUNTRIES,IS_CONSOLIDATOR,IMPORTER_NUMBERED_SHARE,VENDOR_NUMBERED_SHARE,ROUND_1000_SHARE,ROUND_100_SHARE,ANY_HIGH_RISK,HIGH_RISK_SHARE,MAX_GAP_DAYS,FIRST_TXN_VALUE_CAD
0,744129321,4693,1.361284e+06,290.066814,3557.84000,2025-01-02,2025-12-31,4749,6,3,3,3,1,0,0.000211,0.000211,0.000000,0.000000,0,0.000000,3.0,115.330000
1,727341737,78,3.316313e+06,42516.829672,133064.15000,2025-01-05,2025-12-16,86,4,4,4,7,2,0,0.000000,0.000000,0.000000,0.000000,1,0.965116,22.0,68204.310000
2,791945025,23,9.935925e+03,431.996742,919.75744,2025-06-30,2025-12-02,36,8,6,4,2,1,0,0.000000,0.000000,0.000000,0.055556,0,0.000000,89.0,51.515968
3,744202078,21,2.440021e+04,1161.914752,1940.91840,2025-01-03,2025-12-29,32,8,6,4,1,1,0,0.000000,0.000000,0.000000,0.125000,0,0.000000,93.0,571.903200
4,745864801,49,1.065985e+06,21754.790104,528041.67000,2025-01-20,2025-12-30,158,33,5,4,10,7,0,0.000000,0.000000,0.177215,0.443038,0,0.000000,21.0,9593.935000


### Benford's Law tests (I4 / I5)
- Compares each importer's distribution of leading and second digits of **declared transaction value** (`TXN_VALUE_DECL`) against Benford's expected distribution.
- Grain is **one observation per transaction** (`CLNT_SPLY_REQ_ID`), not per line item.

---

- Only single-currency transactions are included (`TXN_VALUE_DECL` is null when `IS_MIXED_CCY_TXN = 1`).
- Non-CAD single-currency transactions are tested in their **native declared units** (USD totals tested as USD, etc.) and pooled per importer.
- Deviation is measured with Mean Absolute Deviation (MAD); larger MAD = less natural-looking distribution.
- Only importers with enough transactions are tested (`BENFORD_MIN_TXNS = 50`).
- The 2nd-digit test uses `TXN_VALUE_DECL >= 10` so every included value has at least two integer digits before the decimal (values 1-9 have no meaningful 2nd digit).
- Official flags: `I4_BENFORD1` / `I5_BENFORD2` (declared). Reference columns: `BENFORD*_MAD_CAD`, `I4_BENFORD1_CAD`, `I5_BENFORD2_CAD` (CAD-converted totals, same rules as before the declared switch).

---

- **Why declared vs CAD Benford can differ:** mixed-currency transactions are excluded from declared but included in CAD (each line converted then summed); FX-missing transactions may have declared totals but null CAD; non-CAD amounts have different digit patterns after conversion.
- **When `DECL_CCY = 'CAD'`:** `TXN_VALUE_DECL` and `TXN_VALUE_CAD` should match for that transaction (FX rate 1.0); any entity-level difference vs CAD Benford comes from pool composition (mixed-ccy / FX-gap exclusions), not from currency mismatch on CAD rows.

In [69]:
def benford_digit_sql(value_col, digit_pos, min_value):
    pos = 1 if digit_pos == 1 else 2
    return f"""
    SELECT BN_9,
    SUBSTR(LTRIM(CAST(CAST(FLOOR({value_col}) AS BIGINT) AS VARCHAR(40))), {pos}, 1) AS D,
    COUNT(*) AS N
    FROM {TXN_TABLE}
    WHERE {value_col} >= {min_value}
    GROUP BY BN_9, SUBSTR(LTRIM(CAST(CAST(FLOOR({value_col}) AS BIGINT) AS VARCHAR(40))), {pos}, 1);
    """

def benford_flags(df_counts, digits, expected, min_txns, mad_threshold):
    df_counts = df_counts.copy()
    df_counts["D"] = df_counts["D"].astype(str)
    df_counts = df_counts[df_counts["D"].isin(digits)]
    piv = df_counts.pivot_table(index="BN_9", columns="D", values="N", aggfunc="sum", fill_value=0)
    piv = piv.reindex(columns=digits, fill_value=0)
    totals = piv.sum(axis=1)
    mad = piv.apply(lambda r: mad_from_counts(r.values, expected), axis=1)
    flag = ((totals >= min_txns) & (mad > mad_threshold)).astype(int)
    return pd.DataFrame({"BN_9": piv.index, "MAD": mad.values, "N": totals.values, "FLAG": flag.values})

digits1 = [str(i) for i in range(1, 10)]
digits2 = [str(i) for i in range(0, 10)]

# Official Benford: declared transaction value (single-currency txns only)
d1_decl = idadb.ida_query(benford_digit_sql("TXN_VALUE_DECL", 1, 1))
d2_decl = idadb.ida_query(benford_digit_sql("TXN_VALUE_DECL", 2, 10))
d1_decl.columns = [c.upper() for c in d1_decl.columns]
d2_decl.columns = [c.upper() for c in d2_decl.columns]
b1_decl = benford_flags(d1_decl, digits1, benford_first_digit_expected(), BENFORD_MIN_TXNS, MAD1_THRESHOLD)
b2_decl = benford_flags(d2_decl, digits2, benford_second_digit_expected(), BENFORD_MIN_TXNS, MAD2_THRESHOLD)
b1_decl = b1_decl.rename(columns={"MAD": "BENFORD1_MAD_DECL", "N": "BENFORD1_N_DECL", "FLAG": "I4_BENFORD1"})
b2_decl = b2_decl.rename(columns={"MAD": "BENFORD2_MAD_DECL", "N": "BENFORD2_N_DECL", "FLAG": "I5_BENFORD2"})

# Reference Benford: CAD transaction value
d1_cad = idadb.ida_query(benford_digit_sql("TXN_VALUE_CAD", 1, 1))
d2_cad = idadb.ida_query(benford_digit_sql("TXN_VALUE_CAD", 2, 10))
d1_cad.columns = [c.upper() for c in d1_cad.columns]
d2_cad.columns = [c.upper() for c in d2_cad.columns]
b1_cad = benford_flags(d1_cad, digits1, benford_first_digit_expected(), BENFORD_MIN_TXNS, MAD1_THRESHOLD)
b2_cad = benford_flags(d2_cad, digits2, benford_second_digit_expected(), BENFORD_MIN_TXNS, MAD2_THRESHOLD)
b1_cad = b1_cad.rename(columns={"MAD": "BENFORD1_MAD_CAD", "N": "BENFORD1_N_CAD", "FLAG": "I4_BENFORD1_CAD"})
b2_cad = b2_cad.rename(columns={"MAD": "BENFORD2_MAD_CAD", "N": "BENFORD2_N_CAD", "FLAG": "I5_BENFORD2_CAD"})

feat = (
    feat.merge(b1_decl, on="BN_9", how="left")
    .merge(b2_decl, on="BN_9", how="left")
    .merge(b1_cad, on="BN_9", how="left")
    .merge(b2_cad, on="BN_9", how="left")
)
print(
    "Benford decl flagged:",
    int(b1_decl["I4_BENFORD1"].sum()),
    "|",
    int(b2_decl["I5_BENFORD2"].sum()),
    "| Benford cad flagged:",
    int(b1_cad["I4_BENFORD1_CAD"].sum()),
    "|",
    int(b2_cad["I5_BENFORD2_CAD"].sum()),
)

Benford decl flagged: 3386 | 3202 | Benford cad flagged: 3408 | 3168


**Interpretation - Benford tests**
- Official flags (`I4`/`I5`) use declared transaction values; CAD versions are reference only (`BENFORD*_MAD_CAD`, `I4_BENFORD1_CAD`, `I5_BENFORD2_CAD`).
- This run: declared flagged **3,386** (I4) / **3,202** (I5); CAD reference flagged **3,408** / **3,168**.
- The gap is mostly mixed-currency exclusions (~25k txns) and FX-missing cases where declared exists but CAD does not.
- A failure is a prompt to look closer, not proof of wrongdoing - fixed price lists or a dominant product can also bend digit distributions.

## Risk-profile matrix
- Turns the eight indicators into flags and one composite score per importer.

---

- Continuous indicators (I8/I9/I10) flag the top of the population (configured percentile); threshold-based indicators use fixed cut-offs; Benford uses MAD cut-offs.
- I9 and I11 are suppressed for consolidators.
- `RISK_SCORE` is the weighted sum of flags (weights default to 1, so score equals flag count).

In [70]:
numeric_cols = [
    "ROUND_1000_SHARE",
    "MAX_GAP_DAYS",
    "FIRST_TXN_VALUE_CAD",
    "IMPORTER_NUMBERED_SHARE",
    "VENDOR_NUMBERED_SHARE",
    "ROUND_100_SHARE",
    "ANY_HIGH_RISK",
    "HIGH_RISK_SHARE",
    "DISTINCT_SECTIONS"
]
for col in numeric_cols:
    feat[col] = pd.to_numeric(feat[col], errors="coerce")

fill_zero = [
    "IMPORTER_NUMBERED_SHARE", "VENDOR_NUMBERED_SHARE", "ROUND_1000_SHARE", "ROUND_100_SHARE",
    "ANY_HIGH_RISK", "HIGH_RISK_SHARE", "MAX_GAP_DAYS", "DISTINCT_SECTIONS", "FIRST_TXN_VALUE_CAD"
]
feat[fill_zero] = feat[fill_zero].fillna(0)
feat["I4_BENFORD1"] = feat["I4_BENFORD1"].fillna(0).astype(int)
feat["I5_BENFORD2"] = feat["I5_BENFORD2"].fillna(0).astype(int)

non_consol = feat["IS_CONSOLIDATOR"] == 0
round_thr = max(feat.loc[non_consol, "ROUND_1000_SHARE"].quantile(FLAG_PCTL), ROUND_SHARE_FLOOR)
gap_thr = feat.loc[feat["TXN_CNT"] >= 2, "MAX_GAP_DAYS"].quantile(FLAG_PCTL)
first_thr = feat["FIRST_TXN_VALUE_CAD"].quantile(FLAG_PCTL)

feat["I3_NUMBERED"] = (
    (feat["IMPORTER_NUMBERED_SHARE"] >= NUMBERED_SHARE_MIN) |
    (feat["VENDOR_NUMBERED_SHARE"] >= NUMBERED_SHARE_MIN)
).astype(int)
feat["I7_HIGH_RISK_JURIS"] = (feat["ANY_HIGH_RISK"] == 1).astype(int)
feat["I8_DORMANCY"] = (
    (feat["MAX_GAP_DAYS"] >= gap_thr) & (feat["TXN_CNT"] >= 2)
).astype(int)
feat["I9_ROUNDED"] = (
    (feat["ROUND_1000_SHARE"] >= round_thr) & non_consol
).astype(int)
feat["I10_HIGH_FIRST_TXN"] = (feat["FIRST_TXN_VALUE_CAD"] >= first_thr).astype(int)
feat["I11_LINES_OF_TRADE"] = (
    (feat["DISTINCT_SECTIONS"] >= SECTIONS_FLAG_MIN) & non_consol
).astype(int)

flag_cols = list(INDICATOR_WEIGHTS.keys())
feat["FLAG_COUNT"] = feat[flag_cols].sum(axis=1)
feat["RISK_SCORE"] = sum(feat[c] * w for c, w in INDICATOR_WEIGHTS.items())
print("thresholds -> round_1000:", round(round_thr, 4), "| gap_days:", round(float(gap_thr), 1), "| first_txn_cad:",
      round(float(first_thr), 2))

thresholds -> round_1000: 0.1 | gap_days: 91.0 | first_txn_cad: 22779.87


In [71]:
show_cols = ["BN_9", "IS_CONSOLIDATOR", "TXN_CNT", "TOTAL_VALUE_CAD", "FIRST_TXN_VALUE_CAD", "MAX_GAP_DAYS"] + flag_cols + ["FLAG_COUNT", "RISK_SCORE"]
top = feat.sort_values(["RISK_SCORE", "TOTAL_VALUE_CAD"], ascending=False).head(25)
display_full(format_specific_columns(top[show_cols], ["TXN_CNT", "TOTAL_VALUE_CAD", "FIRST_TXN_VALUE_CAD"]))

,BN_9,IS_CONSOLIDATOR,TXN_CNT,TOTAL_VALUE_CAD,FIRST_TXN_VALUE_CAD,MAX_GAP_DAYS,I3_NUMBERED,I4_BENFORD1,I5_BENFORD2,I7_HIGH_RISK_JURIS,I8_DORMANCY,I9_ROUNDED,I10_HIGH_FIRST_TXN,I11_LINES_OF_TRADE,FLAG_COUNT,RISK_SCORE
3815,829120914,0,185,"100,397,579.01","359,559.50",16.0,0,1,1,1,0,1,1,1,6,6.0
10617,710252487,0,72,"7,900,132.99","59,595.30",30.0,0,1,1,1,0,1,1,1,6,6.0
8553,846398436,0,95,"6,237,345.17","195,203.25",34.0,0,1,1,1,0,1,1,1,6,6.0
12565,764034955,0,59,"3,986,137.21","585,198.38",36.0,0,1,1,1,0,1,1,1,6,6.0
1246,813675600,0,99,"3,708,819","23,300",33.0,0,1,1,1,0,1,1,1,6,6.0
17429,864229331,0,67,"1,475,196.49","42,446.40",150.0,0,1,1,0,1,1,1,1,6,6.0
10145,739865756,0,56,"1,401,578.40","25,301.58",91.0,0,1,1,1,1,0,1,1,6,6.0
4398,757994470,0,54,"914,126.53","45,172.54",26.0,0,1,1,1,0,1,1,1,6,6.0
10029,816276174,0,905,"1,507,232,572.72","425,884.41",5.0,0,1,1,1,0,0,1,1,5,5.0
12155,125737064,0,"2,504","518,458,834.92","424,737.57",4.0,0,1,1,1,0,0,1,1,5,5.0


**Interpretation - top-ranked importers**
- These are the importers tripping the most indicators, sorted by score then total value.
- Each row shows exactly which indicators fired, so a reviewer can see why an importer surfaced before drilling in.

### Save the risk matrix
- Writes the per-importer flags and score to the database so they can be reused and joined with Notebook 2's anomaly profile.

In [72]:
persist_cols = [
    "BN_9", "IS_CONSOLIDATOR", "TXN_CNT", "TOTAL_VALUE_CAD", "AVG_TXN_VALUE_CAD",
    "MAX_TXN_VALUE_CAD", "FIRST_ARRIVAL", "LAST_ARRIVAL", "DISTINCT_HS6",
    "DISTINCT_SECTIONS", "DISTINCT_VENDORS", "DISTINCT_VENDOR_COUNTRIES",
    "IMPORTER_NUMBERED_SHARE", "ROUND_1000_SHARE", "HIGH_RISK_SHARE",
    "MAX_GAP_DAYS", "FIRST_TXN_VALUE_CAD",
    "BENFORD1_MAD_DECL", "BENFORD2_MAD_DECL", "BENFORD1_MAD_CAD", "BENFORD2_MAD_CAD",
    "I4_BENFORD1_CAD", "I5_BENFORD2_CAD"
]
persist_cols = persist_cols + flag_cols + ["FLAG_COUNT", "RISK_SCORE"]
persist_cols = list(dict.fromkeys(persist_cols))

missing = [c for c in persist_cols if c not in feat.columns]
if missing:
    print("Warning: missing columns:", missing)

valid_cols = [c for c in persist_cols if c in feat.columns]
risk_matrix = feat.loc[:, valid_cols].copy()
for mad_col in ["BENFORD1_MAD_DECL", "BENFORD2_MAD_DECL", "BENFORD1_MAD_CAD", "BENFORD2_MAD_CAD"]:
    risk_matrix[mad_col] = risk_matrix[mad_col].fillna(-1)
print("risk_matrix shape:", risk_matrix.shape)

risk_matrix shape: (19366, 33)


In [73]:
if REBUILD:
    df_to_netezza(idadb, risk_matrix, RISK_MATRIX_TABLE)

Bulk-loaded OADM.TBML_T_NRI_RISK_MATRIX_M -> 19366 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_RISK_MATRIX_M.csv


### Flag combination analysis
- Ranks single flags and co-occurring k-flag combinations (k = 2 through 8) by how many importers trip **all** flags in the combo (AND / co-occurrence).
- An importer may also trip other flags; this is not "exactly k flags total" (see `FLAG_COUNT` in Validation for that).
- Population: all importers; % = count / total importers.
- Consolidators are included (5 of ~19,400); I9 and I11 are structurally 0 for consolidators, so they never contribute to combos that require those flags.


In [ ]:
from itertools import combinations

FLAG_LABELS = {
    "I3_NUMBERED": "I3 Numbered",
    "I4_BENFORD1": "I4 Benford 1st",
    "I5_BENFORD2": "I5 Benford 2nd",
    "I7_HIGH_RISK_JURIS": "I7 High Risk Juris",
    "I8_DORMANCY": "I8 Dormancy",
    "I9_ROUNDED": "I9 Rounded",
    "I10_HIGH_FIRST_TXN": "I10 High First Txn",
    "I11_LINES_OF_TRADE": "I11 Lines of Trade",
}

def combo_label(cols):
    return " + ".join(FLAG_LABELS.get(c, c) for c in cols)

n_ent = len(risk_matrix)
flags = risk_matrix[flag_cols].fillna(0).astype(int)

# k = 1: single-flag frequency
single = flags.sum().sort_values(ascending=False).reset_index()
single.columns = ["flag", "n_importers"]
single["label"] = single["flag"].map(FLAG_LABELS)
single["pct"] = (single["n_importers"] / n_ent).round(4)
print("Single flags by count (descending):")
display_full(format_specific_columns(single[["flag", "label", "n_importers", "pct"]], ["n_importers"]))

# k = 2 .. 8: co-occurrence combinations
for k in range(2, len(flag_cols) + 1):
    rows = []
    for combo in combinations(flag_cols, k):
        combo = list(combo)
        n = int(flags[combo].all(axis=1).sum())
        rows.append({
            "k": k,
            "flags": ", ".join(combo),
            "label": combo_label(combo),
            "n_importers": n,
            "pct": n / n_ent,
        })
    combo_df = pd.DataFrame(rows).sort_values(
        ["n_importers", "label"], ascending=[False, True]
    ).reset_index(drop=True)
    print("Co-occurrence combinations (k=%d, %d combos):" % (k, len(combo_df)))
    display_full(format_specific_columns(combo_df, ["n_importers"]))


**Interpretation - flag combinations**
- Single-flag ranking mirrors Validation `flag_rate`; I11 and I10 dominate, I3 is extremely rare.
- Top pairs/triples are usually dominated by the same frequent flags (I11, I10, I8, I4, I5).
- I4 + I5 co-occurrence is expected to rank high (same transaction family, correlated tests).
- k = 7 and k = 8 tables are likely all zeros if max `FLAG_COUNT` in this run is 6 - that is normal, not an error.


### Save subset
- Save a subset of data as csv.

In [75]:
# x = format_specific_columns(top[show_cols], ["TXN_CNT", "TOTAL_VALUE_CAD", "FIRST_TXN_VALUE_CAD"])
# x[x["RISK_SCORE"]==6].to_csv("data_saved/six_flags.csv", index=False)

## Validation & caveats
- Flag rates per indicator, the consolidator count, and the spread of flag counts - a sanity check that no indicator fires far more or less than expected.

**Known limitations (first pass)**
- No `REQ_VERS_NBR` de-duplication; re-transmitted declarations can inflate value and counts.
- No reliable declared quantity/unit-of-measure, so the FATF pricing indicators (I1 Miscalculations, I2 Inconsistent Prices) are deferred.
- The high-risk check uses the vendor's ISO-2 country only; consignee/importer country are not yet included.
- I3 (Numbered Company) is a digit-pattern heuristic, not a registry lookup of numbered companies.
- All thresholds are starting points and should be tuned against known cases.

In [74]:
print("entities scored:", len(feat))
print("consolidators:", int(feat["IS_CONSOLIDATOR"].sum()))
print()
rates = feat[flag_cols].mean().sort_values(ascending=False)
display_full(rates.to_frame("flag_rate").round(4))
print()
display_full(feat["FLAG_COUNT"].value_counts().sort_index().to_frame("n_entities"))

entities scored: 19366
consolidators: 5



,flag_rate
I11_LINES_OF_TRADE,0.3464
I10_HIGH_FIRST_TXN,0.2500
I8_DORMANCY,0.2001
I4_BENFORD1,0.1748
I5_BENFORD2,0.1653
I7_HIGH_RISK_JURIS,0.1401
I9_ROUNDED,0.0509
I3_NUMBERED,0.0009


,n_entities
FLAG_COUNT,
0,6239
1,5672
2,3751
3,2507
4,959
5,230
6,8


**Interpretation - flag rates**
- Percentile-based indicators (I8/I9/I10) flag close to their design rate (~25% for I10).
- I3 (Numbered Company) is very rare; I11 (Lines of Trade) and Benford (I4/I5) are the most common.
- Most importers trip 0-1 indicators; **8** importers trip all six currently reachable flags (score 6) - the priority review tail.